## 1. Problem Statement

This project analyzes student-performance data to identify factors
associated with academic results.

The analysis focuses on study time, attendance, major, assignment
scores, midterm scores, and final scores.

The main questions are:

- What is the overall student performance?
- Which major has the highest average score?
- Is study time associated with academic performance?
- Is attendance associated with academic performance?
- Which students appear to be outliers?

The dataset may not include all factors affecting student performance,
such as previous academic ability, health, sleep, family conditions,
or course difficulty.

## 2. Imports and Project Paths

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
def find_project_root(start_path: Path) -> Path:
    """Find the project directory containing data, src, and notebooks."""

    start_path = start_path.resolve()

    candidates = [start_path, *start_path.parents]

    for candidate in candidates:
        has_data = (candidate / "data").exists()
        has_src = (candidate / "src").exists()
        has_notebooks = (candidate / "notebooks").exists()

        if has_data and has_src and has_notebooks:
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. "
        "Make sure the project contains data/, src/, and notebooks/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)

Current working directory: c:\Users\LOQ\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance\notebooks
Project root: C:\Users\LOQ\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance


In [3]:
RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "student_performance.csv"
)

PROCESSED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "student_performance_cleaned.csv"
)

FIGURES_DIR = (
    PROJECT_ROOT
    / "reports"
    / "figures"
)

SRC_DIR = PROJECT_ROOT / "src"

In [4]:
PROCESSED_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [5]:
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

In [6]:
print("Project root:")
print(PROJECT_ROOT)

print("\nRaw data path:")
print(RAW_DATA_PATH)

print("\nRaw data exists:")
print(RAW_DATA_PATH.exists())

print("\nProcessed data directory:")
print(PROCESSED_DATA_PATH.parent)

print("\nFigures directory:")
print(FIGURES_DIR)

print("\nSource directory:")
print(SRC_DIR)

Project root:
C:\Users\LOQ\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance

Raw data path:
C:\Users\LOQ\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance\data\raw\student_performance.csv

Raw data exists:
True

Processed data directory:
C:\Users\LOQ\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance\data\processed

Figures directory:
C:\Users\LOQ\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance\reports\figures

Source directory:
C:\Users\LOQ\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance\src


## 3. Load the Data
This section loads the raw student_performance into a pandas DataFrame.
The raw data is kept unchanged. All cleaning and transformations will be performed later using copies of the original DataFrame.

In [7]:
print("Raw data path:", RAW_DATA_PATH)
print("File exists:", RAW_DATA_PATH.exists())

Raw data path: C:\Users\LOQ\Documents\uni\MACHINE_LEARNING\python-for-ml\mini_projects\student_performance\data\raw\student_performance.csv
File exists: True


In [8]:
raw_df = pd.read_csv(RAW_DATA_PATH)

In [9]:
print("Data loaded successfully.")
print("Shape:", raw_df.shape)

Data loaded successfully.
Shape: (201, 8)


In [10]:
raw_df.head()

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
0,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6
1,STU002,Male,Software Engineering,5.8,96.9,7.9,7.8,5.7
2,STU003,Female,Software Engineering,13.0,81.1,9.8,9.1,8.0
3,STU004,Other,Software Engineering,13.8,87.7,8.8,8.1,9.6
4,STU005,Male,Computer Science,2.2,55.4,4.9,5.7,3.7


In [11]:
raw_df.columns.tolist()

['student_id',
 'gender',
 'major',
 'study_hours',
 'attendance_rate',
 'assignment_score',
 'midterm_score',
 'final_score']

In [12]:
type(raw_df)

pandas.DataFrame

## 4. Initial Data Inspection 
This section examines the dataset structure, column types, missing values, duplicate records, numerical summaries, and categorical values.
No data is modified in this section.

In [13]:
print("Number of rows:", raw_df.shape[0])
print("Number of columns:", raw_df.shape[1])

Number of rows: 201
Number of columns: 8


In [14]:
raw_df.head()


,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
0,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6
1,STU002,Male,Software Engineering,5.8,96.9,7.9,7.8,5.7
2,STU003,Female,Software Engineering,13.0,81.1,9.8,9.1,8.0
3,STU004,Other,Software Engineering,13.8,87.7,8.8,8.1,9.6
4,STU005,Male,Computer Science,2.2,55.4,4.9,5.7,3.7


In [15]:
raw_df.tail()

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
196,STU197,Other,Computer Science,12.6,89.1,8.8,8.6,9.2
197,STU198,Other,Computer Science,8.4,82.8,9.3,7.6,7.3
198,STU199,Other,Information Systems,10.0,86.4,8.8,8.5,8.2
199,STU200,Female,Information Systems,9.3,71.5,4.5,6.5,2.5
200,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6


In [16]:
raw_df.dtypes

student_id              str
gender                  str
major                   str
study_hours         float64
attendance_rate     float64
assignment_score    float64
midterm_score       float64
final_score         float64
dtype: object

In [17]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        201 non-null    str    
 1   gender            201 non-null    str    
 2   major             201 non-null    str    
 3   study_hours       201 non-null    float64
 4   attendance_rate   201 non-null    float64
 5   assignment_score  200 non-null    float64
 6   midterm_score     201 non-null    float64
 7   final_score       201 non-null    float64
dtypes: float64(5), str(3)
memory usage: 12.7 KB


In [18]:
raw_df.describe().T

,count,mean,std,min,25%,50%,75%,max
study_hours,201.0,9.816418,3.635420,-3.0,7.4,9.8,12.1,21.7
attendance_rate,201.0,80.119403,11.800431,49.2,71.6,80.9,87.7,105.0
assignment_score,200.0,7.054000,1.713641,1.5,5.9,7.2,8.2,10.0
midterm_score,201.0,7.083085,1.701268,1.8,5.9,7.0,8.4,10.0
final_score,201.0,6.941294,1.997908,-1.0,5.5,7.0,8.6,10.0


In [19]:
raw_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
student_id,201,200,STU001,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,201,4,Other,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN
major,201,4,Software Engineering,86,NaN,NaN,NaN,NaN,NaN,NaN,NaN
study_hours,201.0,NaN,NaN,NaN,9.816418,3.63542,-3.0,7.4,9.8,12.1,21.7
attendance_rate,201.0,NaN,NaN,NaN,80.119403,11.800431,49.2,71.6,80.9,87.7,105.0
assignment_score,200.0,NaN,NaN,NaN,7.054,1.713641,1.5,5.9,7.2,8.2,10.0
midterm_score,201.0,NaN,NaN,NaN,7.083085,1.701268,1.8,5.9,7.0,8.4,10.0
final_score,201.0,NaN,NaN,NaN,6.941294,1.997908,-1.0,5.5,7.0,8.6,10.0


In [20]:
raw_df.isna().sum()

student_id          0
gender              0
major               0
study_hours         0
attendance_rate     0
assignment_score    1
midterm_score       0
final_score         0
dtype: int64

In [21]:
raw_df.duplicated().sum()

np.int64(1)

In [22]:
raw_df[
    raw_df.duplicated(keep=False)
]

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
0,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6
200,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6


In [23]:
raw_df["student_id"].duplicated().sum()

np.int64(1)

In [24]:
raw_df[
    raw_df["student_id"].duplicated(keep=False)
].sort_values("student_id")

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
0,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6
200,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6


In [25]:
raw_df["gender"].value_counts(dropna=False)
raw_df["major"].value_counts(dropna=False)

major
Software Engineering    86
Computer Science        58
Information Systems     56
 computer science        1
Name: count, dtype: int64

In [26]:
raw_df[
    raw_df["study_hours"] < 0
]

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
45,STU046,Male,Computer Science,-3.0,64.6,6.6,4.1,6.4


In [27]:
raw_df[
    ~raw_df["attendance_rate"].between(0, 100)
]

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
12,STU013,Male,Software Engineering,10.3,105.0,5.5,6.1,5.5


In [28]:
score_columns = [
    "assignment_score",
    "midterm_score",
    "final_score",
]

for column in score_columns:
    invalid_rows = raw_df[
        raw_df[column].notna()
        & ~raw_df[column].between(0, 10)
    ]

    print(f"{column}: {len(invalid_rows)} invalid rows")
    display(invalid_rows)

assignment_score: 0 invalid rows


,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score


midterm_score: 0 invalid rows


,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score


final_score: 1 invalid rows


,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
30,STU031,Other,Software Engineering,18.6,62.4,7.4,7.0,-1.0


### Initial Inspection Findings

- The dataset contains 201 rows and 8 columns.
- One exact duplicate row was detected.
- One missing value was found in `assignment_score`.
- An invalid negative value was found in `study_hours`.
- An attendance value greater than 100 was detected.
- An invalid negative value was found in `final_score`.
- Some categorical values contain extra whitespace or inconsistent capitalization.

## 5. Data Validation

This section validates the raw dataset using reusable rules.

The validation process checks required columns, missing values,
duplicate records, duplicate student IDs, invalid numerical ranges,
and inconsistent categorical values.

No data is modified during validation.

In [29]:
from data_utils import validate_student_data

In [30]:
validation_report_before = validate_student_data(raw_df)


In [31]:
validation_report_before

{'is_valid': False,
 'row_count': 201,
 'column_count': 8,
 'missing_columns': [],
 'missing_values': {'student_id': 0,
  'gender': 0,
  'major': 0,
  'study_hours': 0,
  'attendance_rate': 0,
  'assignment_score': 1,
  'midterm_score': 0,
  'final_score': 0},
 'exact_duplicates': 1,
 'duplicate_student_ids': 1,
 'invalid_study_hours': 1,
 'invalid_attendance': 1,
 'invalid_scores': {'assignment_score': 0,
  'midterm_score': 0,
  'final_score': 1}}

In [32]:
general_validation = pd.Series({
    "is_valid": validation_report_before["is_valid"],
    "row_count": validation_report_before["row_count"],
    "column_count": validation_report_before["column_count"],
    "exact_duplicates": (
        validation_report_before["exact_duplicates"]
    ),
    "duplicate_student_ids": (
        validation_report_before["duplicate_student_ids"]
    ),
    "invalid_study_hours": (
        validation_report_before["invalid_study_hours"]
    ),
    "invalid_attendance": (
        validation_report_before["invalid_attendance"]
    ),
})

general_validation

is_valid                 False
row_count                  201
column_count                 8
exact_duplicates             1
duplicate_student_ids        1
invalid_study_hours          1
invalid_attendance           1
dtype: object

In [33]:
missing_validation = pd.Series(
    validation_report_before["missing_values"],
    name="missing_count",
).to_frame()

missing_validation

,missing_count
student_id,0
gender,0
major,0
study_hours,0
attendance_rate,0
assignment_score,1
midterm_score,0
final_score,0


In [34]:
score_validation = pd.Series(
    validation_report_before["invalid_scores"],
    name="invalid_count",
).to_frame()

score_validation

,invalid_count
assignment_score,0
midterm_score,0
final_score,1


In [35]:
raw_df[
    raw_df["assignment_score"].isna()
]

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
5,STU006,Female,Software Engineering,4.8,79.4,NaN,5.5,4.1


In [36]:
raw_df[
    raw_df.duplicated(keep=False)
].sort_values("student_id")

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
0,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6
200,STU001,Female,Software Engineering,11.2,84.1,7.4,7.8,9.6


In [37]:
raw_df[
    raw_df["study_hours"].notna()
    & (raw_df["study_hours"] < 0)
]

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
45,STU046,Male,Computer Science,-3.0,64.6,6.6,4.1,6.4


In [38]:
raw_df[
    raw_df["attendance_rate"].notna()
    & ~raw_df["attendance_rate"].between(0, 100)
]

,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score
12,STU013,Male,Software Engineering,10.3,105.0,5.5,6.1,5.5


In [39]:

from data_utils import SCORE_COLUMNS

for column in SCORE_COLUMNS:
    invalid_rows = raw_df[
        raw_df[column].notna()
        & ~raw_df[column].between(0, 10)
    ]

    print(
        f"{column}: "
        f"{len(invalid_rows)} invalid row(s)"
    )

    display(
        invalid_rows[
            [
                "student_id",
                column,
            ]
        ]
    )

assignment_score: 0 invalid row(s)


,student_id,assignment_score


midterm_score: 0 invalid row(s)


,student_id,midterm_score


final_score: 1 invalid row(s)


,student_id,final_score
30,STU031,-1.0


In [40]:
categorical_columns = [
    "gender",
    "major",
]

for column in categorical_columns:
    has_extra_whitespace = (
        raw_df[column].astype("string")
        != raw_df[column].astype("string").str.strip()
    )

    print(
        f"{column}: "
        f"{has_extra_whitespace.sum()} value(s) "
        "with extra whitespace"
    )

    display(
        raw_df.loc[
            has_extra_whitespace,
            ["student_id", column],
        ]
    )

gender: 1 value(s) with extra whitespace


,student_id,gender
60,STU061,female


major: 1 value(s) with extra whitespace


,student_id,major
20,STU021,computer science


In [41]:
for column in categorical_columns:
    print(f"\nColumn: {column}")

    display(
        raw_df[column]
        .value_counts(dropna=False)
        .to_frame(name="count")
    )


Column: gender


,count
gender,
Other,70
Female,65
Male,65
female,1



Column: major


,count
major,
Software Engineering,86
Computer Science,58
Information Systems,56
computer science,1


### Validation Findings

The raw dataset is not yet valid for analysis.

The validation process detected the following issues:

- One missing value exists in `assignment_score`.
- One exact duplicate row exists.
- One duplicated `student_id` exists.
- One negative value exists in `study_hours`.
- One attendance value is greater than 100.
- One negative value exists in `final_score`.
- Some categorical values contain extra whitespace.
- Some categorical values use inconsistent capitalization.

These issues will be addressed in the Data Cleaning section.

## 6. Data Cleaning

This section creates a cleaned copy of the raw dataset.

Cleaning includes removing duplicates, normalizing categorical values,
converting numerical columns, handling invalid ranges, and filling
missing numerical values with column medians.

The original raw DataFrame is not modified.

In [42]:
from data_utils import clean_student_data

In [43]:
raw_snapshot = raw_df.copy(deep=True)

In [44]:
clean_df = clean_student_data(raw_df)

In [45]:
print("Raw shape:", raw_df.shape)
print("Clean shape:", clean_df.shape)

Raw shape: (201, 8)
Clean shape: (200, 8)


In [46]:
pd.testing.assert_frame_equal(
    raw_df,
    raw_snapshot,
)

print("raw_df was not modified.")

raw_df was not modified.


In [47]:
validation_report_after = validate_student_data(
    clean_df
)

validation_report_after

{'is_valid': True,
 'row_count': 200,
 'column_count': 8,
 'missing_columns': [],
 'missing_values': {'student_id': 0,
  'gender': 0,
  'major': 0,
  'study_hours': 0,
  'attendance_rate': 0,
  'assignment_score': 0,
  'midterm_score': 0,
  'final_score': 0},
 'exact_duplicates': 0,
 'duplicate_student_ids': 0,
 'invalid_study_hours': 0,
 'invalid_attendance': 0,
 'invalid_scores': {'assignment_score': 0,
  'midterm_score': 0,
  'final_score': 0}}

In [48]:
print(
    "Missing values:",
    clean_df.isna().sum().sum(),
)

print(
    "Exact duplicates:",
    clean_df.duplicated().sum(),
)

print(
    "Duplicate IDs:",
    clean_df["student_id"].duplicated().sum(),
)

Missing values: 0
Exact duplicates: 0
Duplicate IDs: 0


### Cleaning Decisions

- Exact duplicate rows were removed.
- Duplicate student IDs were removed while keeping the first record.
- Student IDs were stripped and converted to uppercase.
- Gender and major values were stripped and converted to title case.
- Invalid numerical values were converted to missing values.
- Missing numerical values were filled using the median of each column.
- The original raw dataset was not modified.

### 7. Feature Engineering
This section creates new features for analysis:
- `average_score`: weighted average of assignment, midterm, and final scores.
- `passed`: whether the weighted average is at least 5.
- `study_level`: categorical grouping based on study hours.
- `score_improvement`: difference between final and midterm scores.

In [49]:
from data_utils import create_score_features

In [50]:
import inspect

print(inspect.getsource(create_score_features))

def create_score_features(
    df: pd.DataFrame,
) -> pd.DataFrame:


    featured_df = df.copy(deep=True)

    featured_df["average_score"] = (
        featured_df["assignment_score"] * 0.2
        + featured_df["midterm_score"] * 0.3
        + featured_df["final_score"] * 0.5
    ).round(2)

    featured_df["passed"] = (
        featured_df["average_score"] >= 5
    )

    featured_df["study_level"] = pd.cut(
        featured_df["study_hours"],
        bins=[0, 5, 10, 20, np.inf],
        labels=[
            "Low",
            "Moderate",
            "High",
            "Very high",
        ],
        include_lowest=True,
    )

    featured_df["score_improvement"] = (
        featured_df["final_score"]
        - featured_df["midterm_score"]
    ).round(2)

    return featured_df



In [51]:
import importlib
import data_utils

importlib.reload(data_utils)

<module 'data_utils' from 'C:\\Users\\LOQ\\Documents\\uni\\MACHINE_LEARNING\\python-for-ml\\mini_projects\\student_performance\\src\\data_utils.py'>

In [52]:
analysis_df = data_utils.create_score_features(clean_df)

In [53]:
import inspect

print(
    inspect.getsource(
        data_utils.create_score_features
    )
)

def create_score_features(
    df: pd.DataFrame,
) -> pd.DataFrame:


    featured_df = df.copy(deep=True)

    featured_df["average_score"] = (
        featured_df["assignment_score"] * 0.2
        + featured_df["midterm_score"] * 0.3
        + featured_df["final_score"] * 0.5
    ).round(2)

    featured_df["passed"] = (
        featured_df["average_score"] >= 5
    )

    featured_df["study_level"] = pd.cut(
        featured_df["study_hours"],
        bins=[0, 5, 10, 20, np.inf],
        labels=[
            "Low",
            "Moderate",
            "High",
            "Very high",
        ],
        include_lowest=True,
    )

    featured_df["score_improvement"] = (
        featured_df["final_score"]
        - featured_df["midterm_score"]
    ).round(2)

    return featured_df



In [54]:
analysis_df[
    [
        "student_id",
        "average_score",
        "passed",
        "study_level",
        "score_improvement",
    ]
].head()

,student_id,average_score,passed,study_level,score_improvement
0,STU001,8.62,True,High,1.8
1,STU002,6.77,True,Moderate,-2.1
2,STU003,8.69,True,High,-1.1
3,STU004,8.99,True,High,1.5
4,STU005,4.54,False,Low,-2.0


In [55]:
analysis_df[
    [
        "average_score",
        "score_improvement",
    ]
].describe()

,average_score,score_improvement
count,200.000000,200.000000
mean,7.018450,-0.111500
std,1.638372,1.461712
min,2.270000,-4.400000
25%,5.837500,-1.100000
50%,6.855000,0.000000
75%,8.332500,0.900000
max,10.000000,3.000000


In [56]:
analysis_df["passed"].value_counts()

passed
True     177
False     23
Name: count, dtype: int64

## 8. Exploratory Data Analysis

This section answers the main analytical questions using the cleaned
and feature-engineered dataset.

### Câu 1: Điểm trung bình


In [57]:
overall_average = (analysis_df["average_score"].mean())
print("Overall average score:", overall_average)

Overall average score: 7.0184500000000005


### Câu 2: Tỷ lệ đậu


In [58]:
pass_rate = (analysis_df["passed"].mean()*100)
print("Pass rate: {:.2f}%".format(pass_rate))

Pass rate: 88.50%


### Câu 3: Ngành có điểm trung bình cao nhất

In [59]:
major_average = (
    analysis_df.groupby("major")["average_score"].mean().sort_values(ascending = False)
)

major_average

major
Computer Science        7.092542
Information Systems     7.087143
Software Engineering    6.921765
Name: average_score, dtype: float64

In [60]:
highest_major = major_average.idxmax()

print("Major with the highest average score:", highest_major, )

Major with the highest average score: Computer Science


### Câu 4: Ngành có tỷ lệ đậu thấp nhất

In [61]:
major_pass_rate = (
    analysis_df.groupby("major")["passed"].mean().mul(100).sort_values()
)

major_pass_rate

major
Information Systems     87.500000
Software Engineering    88.235294
Computer Science        89.830508
Name: passed, dtype: float64

In [62]:
lowest_pass_major = major_pass_rate.idxmin()

print("Major with lowest pass rate:", lowest_pass_major,)

Major with lowest pass rate: Information Systems


### Câu 5: Study hours liên hệ với kết quả như thế nào?

#### Tính tương quan

In [63]:
study_score_correlation = (
    analysis_df[
        ["study_hours", "average_score"]
    ]
    .corr()
    .loc[
        "study_hours", "average_score"
    ]
)

print("Study_hours correlation:", round(study_score_correlation, 3))

Study_hours correlation: 0.472


#### So sánh theo nhóm study level

In [64]:
study_level_summary = (
    analysis_df.groupby("study_level", observed=True)
    .agg(
        student_count=("student_id", "count"),
        average_score=("average_score", "mean"),
        pass_rate=("passed", "mean"),
    )
)

study_level_summary["pass_rate"] *= 100

study_level_summary


,student_count,average_score,pass_rate
study_level,,,
Low,16,5.575000,68.750000
Moderate,90,6.581667,83.333333
High,93,7.660860,96.774194
Very high,1,9.680000,100.000000


### Câu 6: Attendance liên hệ với kết quả như thế nào

In [65]:
attendace_score_correlation = (
    analysis_df[
        ["attendance_rate", "average_score"]
    ]
    .corr()
    .loc[
        "attendance_rate", "average_score"  
    ]
)

print("Attendance_rate correlation:", round(attendace_score_correlation, 3))

Attendance_rate correlation: 0.292


### Câu 7: Có outlier về điểm không

In [66]:
from data_utils import find_iqr_outliers

In [67]:
score_outliers = find_iqr_outliers(
    analysis_df,
    "average_score",
)

print(
    "Number of average-score outliers:",
    len(score_outliers),
)

score_outliers

Number of average-score outliers: 0


,student_id,gender,major,study_hours,attendance_rate,assignment_score,midterm_score,final_score,average_score,passed,study_level,score_improvement


### Câu 8: Phân phối giữa các nhóm có khác nhau không

#### So sánh theo giới tính

In [68]:
gender_summary = (
    analysis_df
    .groupby("gender")
    .agg(
        student_count=(
            "student_id",
            "count",
        ),
        mean_score=(
            "average_score",
            "mean",
        ),
        median_score=(
            "average_score",
            "median",
        ),
        score_std=(
            "average_score",
            "std",
        ),
        pass_rate=(
            "passed",
            "mean",
        ),
    )
)

gender_summary["pass_rate"] *= 100

gender_summary

,student_count,mean_score,median_score,score_std,pass_rate
gender,,,,,
Female,65,6.853846,6.650,1.704186,86.153846
Male,65,6.976308,6.770,1.550990,89.230769
Other,70,7.210429,7.155,1.658895,90.000000


#### So sánh đồng thời theo ngành và giới tính

In [69]:
group_summary = (
    analysis_df
    .groupby(
        [
            "major",
            "gender",
        ]
    )
    .agg(
        student_count=(
            "student_id",
            "count",
        ),
        average_score=(
            "average_score",
            "mean",
        ),
        pass_rate=(
            "passed",
            "mean",
        ),
    )
)

group_summary["pass_rate"] *= 100

group_summary

student_count  average_score   pass_rate
major                gender                                          
Computer Science     Female             10       7.144000   90.000000
                     Male               28       6.888571   89.285714
                     Other              21       7.340000   90.476190
Information Systems  Female             25       7.019200   80.000000
                     Male               16       7.690625  100.000000
                     Other              15       6.556667   86.666667
Software Engineering Female             30       6.619333   90.000000
                     Male               21       6.549048   80.952381
                     Other              34       7.418824   91.176471